# Vehicle Make Classifier — EfficientNet-B0

Train a vehicle make classifier to verify if the car in the image matches the claimed make.

**Dataset:** Stanford Cars (196 classes) → grouped into ~30 makes
**Model:** EfficientNet-B0 (pretrained on ImageNet)
**Output:** `vehicle_make_classifier.pt`

**Setup:** GPU T4, Internet On, Persistence Files

In [ ]:
!pip install -q datasets transformers timm pillow torchvision

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from datasets import load_dataset
from PIL import Image
import numpy as np
from collections import Counter
import json
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Load Stanford Cars & Extract Makes

In [ ]:
# Load Stanford Cars from HuggingFace
ds = load_dataset("tanganke/stanford_cars")
print(f"Train: {len(ds['train'])}, Test: {len(ds['test'])}")
print(f"Example label: {ds['train'][0]['label']}")
print(f"Class names sample: {ds['train'].features['label'].names[:10]}")

In [ ]:
# Get all class names
class_names = ds['train'].features['label'].names
print(f"Total fine-grained classes: {len(class_names)}")

# Extract make (first word(s) before model name)
# Stanford Cars format: "Make Model Year" e.g. "BMW M3 Coupe 2012"
def extract_make(class_name):
    """Extract vehicle make from Stanford Cars class name."""
    name = class_name.lower().strip()
    
    # Multi-word makes
    multi_word_makes = [
        "aston martin", "land rover", "rolls-royce", "rolls royce",
        "am general", "plymouth fury",
    ]
    for make in multi_word_makes:
        if name.startswith(make):
            return make.replace("-", " ")
    
    # Single word make (first word)
    first_word = name.split()[0]
    return first_word

# Map each fine-grained class to its make
class_to_make = {i: extract_make(name) for i, name in enumerate(class_names)}

# Get unique makes
all_makes = sorted(set(class_to_make.values()))
make_to_idx = {make: i for i, make in enumerate(all_makes)}
idx_to_make = {i: make for make, i in make_to_idx.items()}

print(f"\nExtracted {len(all_makes)} unique makes:")
for i, make in enumerate(all_makes):
    count = sum(1 for v in class_to_make.values() if v == make)
    print(f"  {i:2d}. {make} ({count} sub-classes)")

In [ ]:
# Check distribution of makes in training set
train_makes = [class_to_make[item['label']] for item in ds['train']]
make_counts = Counter(train_makes)
print("Training samples per make:")
for make, count in make_counts.most_common():
    print(f"  {make}: {count}")

## 2. Create Dataset & DataLoaders

In [ ]:
# Image transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class CarsDataset(Dataset):
    def __init__(self, hf_dataset, class_to_make, make_to_idx, transform):
        self.dataset = hf_dataset
        self.class_to_make = class_to_make
        self.make_to_idx = make_to_idx
        self.transform = transform
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image'].convert('RGB')
        fine_label = item['label']
        make = self.class_to_make[fine_label]
        make_idx = self.make_to_idx[make]
        
        image = self.transform(image)
        return image, make_idx


train_dataset = CarsDataset(ds['train'], class_to_make, make_to_idx, train_transform)
val_dataset = CarsDataset(ds['test'], class_to_make, make_to_idx, val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"Num classes (makes): {len(all_makes)}")

## 3. Build Model

In [ ]:
# Use EfficientNet-B0 (lightweight, fast inference)
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=len(all_makes))
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params / 1e6:.1f}M")
print(f"Trainable: {trainable_params / 1e6:.1f}M")

## 4. Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

num_epochs = 30
best_val_acc = 0.0
best_model_state = None

for epoch in range(num_epochs):
    # Train
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
    
    scheduler.step()
    train_acc = train_correct / train_total
    
    # Validate
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = val_correct / val_total
    
    print(f"Epoch {epoch+1:2d}/{num_epochs} | "
          f"Train Loss: {train_loss/train_total:.4f} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Val Acc: {val_acc:.4f} | "
          f"LR: {scheduler.get_last_lr()[0]:.6f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        print(f"  → New best: {val_acc:.4f}")

print(f"\n✅ Best validation accuracy: {best_val_acc:.4f}")

## 5. Evaluate

In [ ]:
# Load best model
model.load_state_dict(best_model_state)
model.eval()

# Per-class accuracy
class_correct = {make: 0 for make in all_makes}
class_total = {make: 0 for make in all_makes}

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        
        for pred, label in zip(predicted.cpu().numpy(), labels.cpu().numpy()):
            make = idx_to_make[label]
            class_total[make] += 1
            if pred == label:
                class_correct[make] += 1

print("Per-make accuracy:")
print(f"{'Make':<20} {'Acc':>6} {'Samples':>8}")
print("-" * 36)
for make in sorted(all_makes):
    if class_total[make] > 0:
        acc = class_correct[make] / class_total[make]
        print(f"{make:<20} {acc:>6.1%} {class_total[make]:>8}")

## 6. Export Model

In [ ]:
# Save complete model package
save_dict = {
    'model_state_dict': best_model_state,
    'model_name': 'efficientnet_b0',
    'num_classes': len(all_makes),
    'class_names': all_makes,  # list of make names
    'make_to_idx': make_to_idx,
    'idx_to_make': idx_to_make,
    'best_val_acc': best_val_acc,
    'input_size': 224,
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225],
}

output_path = '/kaggle/working/vehicle_make_classifier.pt'
torch.save(save_dict, output_path)

print(f"✅ Model saved to: {output_path}")
print(f"   Size: {os.path.getsize(output_path) / 1024**2:.1f} MB")
print(f"   Classes: {len(all_makes)} makes")
print(f"   Best val acc: {best_val_acc:.4f}")
print(f"\n📥 Download and place at:")
print(f"   ml/weights/vehicle_make/vehicle_make_classifier.pt")

In [ ]:
# Also save the class mapping as JSON for easy reference
mapping = {
    'makes': all_makes,
    'make_to_idx': make_to_idx,
}
with open('/kaggle/working/vehicle_makes.json', 'w') as f:
    json.dump(mapping, f, indent=2)

print("Saved vehicle_makes.json")
print(json.dumps(mapping['makes'], indent=2))

## 7. Quick Inference Test

In [ ]:
# Test on a few validation images
model.eval()
test_samples = [ds['test'][i] for i in range(10)]

print(f"{'Actual Make':<20} {'Predicted Make':<20} {'Confidence':>10} {'Correct'}")
print("-" * 65)

for sample in test_samples:
    image = sample['image'].convert('RGB')
    actual_make = class_to_make[sample['label']]
    
    input_tensor = val_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        conf, pred_idx = probs.max(1)
    
    pred_make = idx_to_make[pred_idx.item()]
    correct = "✅" if pred_make == actual_make else "❌"
    print(f"{actual_make:<20} {pred_make:<20} {conf.item():>8.1%}   {correct}")